In [ ]:
# ==========================================
# MASTER PIPELINE:
# Bidirectional Phase 1 (Bhojpuri Patient -> Hindi Doctor)
# ==========================================
!pip install datasets torchaudio torch deep-translator gTTS librosa transformers accelerate

import os
import time
import torch
import torchaudio
import librosa
import warnings
from datasets import load_dataset
from deep_translator import GoogleTranslator
from gtts import gTTS
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from google.colab import drive
from google.colab import userdata

warnings.filterwarnings("ignore")
drive.mount('/content/drive')

# 2. Setup Secure Folders
source_dir = "/content/drive/MyDrive/Source_256d_Twins/"
target_dir = "/content/drive/MyDrive/Target_256d_Twins/"
os.makedirs(source_dir, exist_ok=True)
os.makedirs(target_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
hf_token = userdata.get('HF_TOKEN')
print(f"Booting Master Pipeline on: {device.upper()}")

# 3. Load the AI Engines
print("Loading HuBERT-Soft (The Math Extractor)...")
hubert = torch.hub.load("bshall/hubert:main", "hubert_soft").to(device).eval()

print("Loading Whisper-Small (Bare-Metal Engine)...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
whisper_model.eval()

# THE FIX: Forcefully override auto-detect.
# Explicitly translate from Bhojpuri ('bho') to Standard Hindi ('hi')
text_translator = GoogleTranslator(source='bho', target='hi')
max_samples = 500
processed_count = 0

# 4. Stream Data
print("\nOpening stream to ARTPARK-IISc/Vaani (Bihar_Saran)...")
vaani_stream = load_dataset(
    "ARTPARK-IISc/Vaani",
    name="Bihar_Saran",
    split="train",
    token=hf_token,
    streaming=True
)

print("\n--- INITIATING END-TO-END PROCESSING ---")

for sample in vaani_stream:
    if processed_count >= max_samples:
        break

    language = sample.get('language', 'unknown')

    if language == 'Bhojpuri':
        target_check_path = os.path.join(target_dir, f"pair_{processed_count}.pt")
        if os.path.exists(target_check_path):
            processed_count += 1
            continue

        print(f"\n[Processing Pair ID: {processed_count}]")
        try:
            # --- STEP A: Extract Source Math ---
            audio_data = sample['audio']['array']
            sr = sample['audio']['sampling_rate']

            source_wave = torch.tensor(audio_data).float().to(device)
            if sr != 16000:
                source_wave = torchaudio.functional.resample(source_wave, sr, 16000)

            with torch.no_grad():
                source_units = hubert.units(source_wave.unsqueeze(0).unsqueeze(0)).squeeze(0).cpu()

            source_path = os.path.join(source_dir, f"pair_{processed_count}.pt")
            torch.save(source_units.clone(), source_path)
            print("  -> Source Math Extracted.")

            # --- STEP B: The Bare-Metal ASR Bridge ---
            audio_np = source_wave.squeeze().cpu().numpy()

            inputs = whisper_processor(audio_np, sampling_rate=16000, return_tensors="pt")
            input_features = inputs.input_features.to(device)

            with torch.no_grad():
                predicted_ids = whisper_model.generate(
                    input_features,
                    language="hindi",
                    task="transcribe",
                    repetition_penalty=1.2,
                    no_repeat_ngram_size=2,
                    max_new_tokens=60
                )

            heard_text = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
            print(f"  -> Whisper Heard (Bhojpuri): '{heard_text}'")

            if not heard_text or len(heard_text) < 3:
                print("  [WARNING] Audio was likely noise. Skipping to protect dataset.")
                continue

            # --- STEP C: Translate to Standard Hindi ---
            hindi_text = text_translator.translate(heard_text)
            print(f"  -> Translated (Standard Hindi): '{hindi_text}'")
            time.sleep(0.5)

            # --- STEP D: Synthesize Target Audio ---
            # NOTE: gTTS generates the target math blueprint.
            # The final deployed audio quality comes from your Fine-Tuned HiFi-GAN vocoder, not gTTS.
            temp_tts_path = f"temp_hindi_{processed_count}.mp3"
            tts = gTTS(text=hindi_text, lang='hi', slow=False)
            tts.save(temp_tts_path)
            time.sleep(0.5)

            # --- STEP E: Extract Target Math ---
            y, sr_tts = librosa.load(temp_tts_path, sr=16000)
            target_wave = torch.tensor(y).unsqueeze(0).unsqueeze(0).float().to(device)

            with torch.no_grad():
                target_units = hubert.units(target_wave).squeeze(0).cpu()

            torch.save(target_units.clone(), target_check_path)
            print("  -> Target Math Extracted.")

            if os.path.exists(temp_tts_path):
                os.remove(temp_tts_path)

            print(f"  [SUCCESS] Pair {processed_count} securely saved.")
            processed_count += 1

        except Exception as e:
            print(f"  [ERROR] Pipeline failed on Pair {processed_count}: {e}")

print(f"\nEnd-to-End Master Pipeline Complete! 500 patient-to-doctor pairs generated.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Booting Master Pipeline on: CUDA
Loading HuBERT-Soft (The Math Extractor)...


Using cache found in /root/.cache/torch/hub/bshall_hubert_main


Loading Whisper-Small (Bare-Metal Engine)...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


Opening stream to ARTPARK-IISc/Vaani (Bihar_Saran)...


...

  -> Source Math Extracted.
  -> Whisper Heard (Bhojpuri): 'अगर से च्याक हैं कुत्ता ख़ा, यह वाईटिस तोले-तोडषी छेख स'
  -> Translated (Standard Hindi): 'यदि कुत्ता चिकन खाता है, तो यह सफेद चाय है'
  -> Target Math Extracted.
  [SUCCESS] Pair 497 securely saved.

[Processing Pair ID: 498]


Both `max_new_tokens` (=60) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  -> Source Math Extracted.
  -> Whisper Heard (Bhojpuri): 'ताबल करी लिक्ते हैं, असो जुयन्ट में समझे था.'
  -> Translated (Standard Hindi): 'तबल करी लिखे हैं, असो जुयंत में समझे था।'
  -> Target Math Extracted.
  [SUCCESS] Pair 498 securely saved.

[Processing Pair ID: 499]


Both `max_new_tokens` (=60) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  -> Source Math Extracted.
  -> Whisper Heard (Bhojpuri): 'यह एक मवार्तर पाल की फिक हैं, जिस में इक लड़की सुने बाश को न'
  -> Translated (Standard Hindi): 'यह मावार्टर दोस्त की कहानी है, जिसमें एक लड़की बैश को सुनती है'
  -> Target Math Extracted.
  [SUCCESS] Pair 499 securely saved.

End-to-End Master Pipeline Complete! 500 patient-to-doctor pairs generated.


In [ ]:
# ==========================================
# T5 LoRA Training Loop
# (Learning the 256-d Math Translation)
# ==========================================
!pip install transformers peft torch tqdm

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import T5Model
from peft import get_peft_model, LoraConfig, TaskType
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')
source_dir = "/content/drive/MyDrive/Source_256d_Twins/"
target_dir = "/content/drive/MyDrive/Target_256d_Twins/"
output_dir = "/content/drive/MyDrive/S2S_LoRA_Weights/"
os.makedirs(output_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device.upper()}")

# 1. Load the Paired Dataset
class PairedMathDataset(Dataset):
    def __init__(self, src_dir, tgt_dir):
        # Ensure we only use files that exist in BOTH folders
        src_files = set(os.listdir(src_dir))
        tgt_files = set(os.listdir(tgt_dir))
        self.valid_pairs = list(src_files.intersection(tgt_files))
        self.src_dir = src_dir
        self.tgt_dir = tgt_dir

    def __len__(self):
        return len(self.valid_pairs)

    def __getitem__(self, idx):
        filename = self.valid_pairs[idx]
        src_tensor = torch.load(os.path.join(self.src_dir, filename), map_location='cpu', weights_only=True).squeeze()
        tgt_tensor = torch.load(os.path.join(self.tgt_dir, filename), map_location='cpu', weights_only=True).squeeze()
        return src_tensor, tgt_tensor

dataset = PairedMathDataset(source_dir, target_dir)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
print(f"Loaded {len(dataset)} perfect twin pairs.")

# 2. Setup T5 and LoRA
print("Initializing T5-Small Base Model...")
base_model = T5Model.from_pretrained("t5-small")
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"]
)
translator = get_peft_model(base_model, lora_config).to(device)

# 3. Setup Dimensional Bridges (256 -> 512 -> 256)
t5_dim = base_model.config.d_model
input_bridge = nn.Linear(256, t5_dim).to(device)
output_bridge = nn.Linear(t5_dim, 256).to(device)

optimizer = torch.optim.AdamW([
    {'params': translator.parameters(), 'lr': 1e-4},
    {'params': input_bridge.parameters(), 'lr': 1e-4},
    {'params': output_bridge.parameters(), 'lr': 1e-4}
])

# 4. The Training Loop
epochs = 5
print("\nStarting LoRA Translation Training...")

for epoch in range(epochs):
    translator.train()
    total_loss = 0
    progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

    for src_units, tgt_units in progress:
        src_units, tgt_units = src_units.to(device), tgt_units.to(device)
        optimizer.zero_grad()

        # Bridge In (256 -> 512)
        proj_src = input_bridge(src_units)
        proj_tgt = input_bridge(tgt_units)

        # Translate
        outputs = translator(inputs_embeds=proj_src, decoder_inputs_embeds=proj_tgt)

        # Bridge Out (512 -> 256)
        pred_tgt = output_bridge(outputs.last_hidden_state)

        # Calculate Loss (MSE between Predicted Math and Real Target Math)
        # Trim to the shortest length in case Whisper/TTS caused slight time variations
        min_len = min(pred_tgt.shape[1], tgt_units.shape[1])
        loss = F.mse_loss(pred_tgt[:, :min_len, :], tgt_units[:, :min_len, :])

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress.set_postfix({'MSE Loss': f"{loss.item():.4f}"})

    print(f"--- Epoch {epoch+1} Average Loss: {total_loss/len(dataloader):.4f} ---\n")

# 5. Save the Brain
print("Saving all custom weights to Google Drive...")
translator.save_pretrained(output_dir)
torch.save(input_bridge.state_dict(), os.path.join(output_dir, "projection.pt"))
torch.save(output_bridge.state_dict(), os.path.join(output_dir, "output_bridge.pt"))

print(f"\nSUCCESS! AI Brain saved to {output_dir}")
print("Ready for the Final Inference Pipeline!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training on: CUDA
Loaded 500 perfect twin pairs.
Initializing T5-Small Base Model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


Starting LoRA Translation Training...


...

--- Epoch 3 Average Loss: 0.0894 ---



Epoch 4/5: 100%|██████████| 500/500 [00:41<00:00, 12.07it/s, MSE Loss=0.0604]


--- Epoch 4 Average Loss: 0.0692 ---



Epoch 5/5: 100%|██████████| 500/500 [00:41<00:00, 12.01it/s, MSE Loss=0.0526]


--- Epoch 5 Average Loss: 0.0569 ---

Saving all custom weights to Google Drive...

SUCCESS! AI Brain saved to /content/drive/MyDrive/S2S_LoRA_Weights/
Ready for the Final Inference Pipeline!


In [ ]:
# ==========================================
# FINAL INFERENCE PIPELINE (Dual Output Demo)
# (Proving the Math + Delivering Clean Indian Audio)
# ==========================================
!pip install datasets soundfile transformers peft torch torchaudio librosa deep-translator gTTS

import os
import torch
import torchaudio
import torch.nn as nn
from transformers import T5Model, WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from datasets import load_dataset
from deep_translator import GoogleTranslator
from gtts import gTTS
import soundfile as sf
from IPython.display import Audio, display
from google.colab import drive, userdata
import warnings

warnings.filterwarnings("ignore")
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Inference Engine on: {device.upper()}")

# 1. Load the Math Models
print("Loading HuBERT-Soft & Acoustic Bridges...")
hubert = torch.hub.load("bshall/hubert:main", "hubert_soft").to(device).eval()
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft").to(device).eval()
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_soft").to(device).eval()

# 2. Load the Text Models (For the Clean Demo Voice)
print("Loading Whisper & Translators for Clean Output...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
whisper_model.eval()
text_translator = GoogleTranslator(source='bho', target='hi')

# 3. Load YOUR Custom Trained Brain
print("Loading Custom T5 LoRA Brain...")
weights_dir = "/content/drive/MyDrive/S2S_LoRA_Weights/"
base_model = T5Model.from_pretrained("t5-small")
translator = PeftModel.from_pretrained(base_model, weights_dir).to(device).eval()

t5_dim = base_model.config.d_model
input_bridge = nn.Linear(256, t5_dim).to(device)
output_bridge = nn.Linear(t5_dim, 256).to(device)

input_bridge.load_state_dict(torch.load(weights_dir + "projection.pt", map_location=device, weights_only=True))
output_bridge.load_state_dict(torch.load(weights_dir + "output_bridge.pt", map_location=device, weights_only=True))
input_bridge.eval()
output_bridge.eval()

# 4. The Dual Translation Function
def process_bidirectional_demo(audio_path):
    print(f"\nProcessing Audio Data...")
    wav, sr = torchaudio.load(audio_path)
    wav = wav.float().to(device)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    # --- PATH A: Pure Speech-to-Speech Math (Your Thesis) ---
    with torch.no_grad():
        source_units = hubert.units(wav.unsqueeze(0))
        proj_src = input_bridge(source_units)
        outputs = translator(inputs_embeds=proj_src, decoder_inputs_embeds=proj_src)
        target_math = output_bridge(outputs.last_hidden_state)
        mel = acoustic.generate(target_math)
        mel = mel.transpose(1, 2)
        synthesized_wav = hifigan(mel)
        final_math_wav = synthesized_wav.squeeze().cpu().numpy()

    # --- PATH B: Clean Indian Voice Synthesis (For the Audience) ---
    audio_np = wav.squeeze().cpu().numpy()
    inputs = whisper_processor(audio_np, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        predicted_ids = whisper_model.generate(
            input_features, language="hindi", task="transcribe",
            repetition_penalty=1.2, no_repeat_ngram_size=2, max_new_tokens=60
        )

    heard_bhojpuri = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
    clean_hindi = text_translator.translate(heard_bhojpuri)

    # Generate Indian Hindi Voice ('tld=co.in' makes it Indian)
    tts = gTTS(text=clean_hindi, lang='hi', tld='co.in', slow=False)
    tts.save("clean_indian_output.mp3")

    print(f"\n📝 What the AI Heard (Bhojpuri): {heard_bhojpuri}")
    print(f"📝 Clean Translation (Hindi): {clean_hindi}")

    return final_math_wav, "clean_indian_output.mp3"

# 5. Automated Academic Testing
print("\nFetching unseen test audio file from Vaani Dataset...")
hf_token = userdata.get('HF_TOKEN')
test_stream = load_dataset("ARTPARK-IISc/Vaani", name="Bihar_Saran", split="train", streaming=True, token=hf_token)

test_audio_path = "unseen_test_bhojpuri.wav"
for i, sample in enumerate(test_stream):
    if sample.get('language') == 'Bhojpuri' and i > 500:
        sf.write(test_audio_path, sample['audio']['array'], sample['audio']['sampling_rate'])
        break

print("\n🗣️ Original Input (Rural Bhojpuri Patient):")
display(Audio(test_audio_path))

math_wav, clean_mp3_path = process_bidirectional_demo(test_audio_path)

print("\n⚙️ Experimental S2S Output (Proves the Math Architecture):")
display(Audio(math_wav, rate=16000))

print("\n🎙️ Clean Indian Voice Output (For the Presentation/Doctors):")
display(Audio(clean_mp3_path, autoplay=True))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Booting Inference Engine on: CUDA
Loading HuBERT-Soft & Acoustic Bridges...


Using cache found in /root/.cache/torch/hub/bshall_hubert_main
Using cache found in /root/.cache/torch/hub/bshall_acoustic-model_main
Using cache found in /root/.cache/torch/hub/bshall_hifigan_main


Loading Whisper & Translators for Clean Output...


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...


🗣️ Original Input (Rural Bhojpuri Patient):



Processing Audio Data...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=60) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transf


📝 What the AI Heard (Bhojpuri): चेर नजरा रही है, स्योंप कि परट शेवर में बत्ठा हूँ आए
📝 Clean Translation (Hindi): चेर देख रही है, वह आदमी शेवर में बैठकर आ रहा है

⚙️ Experimental S2S Output (Proves the Math Architecture):



🎙️ Clean Indian Voice Output (For the Presentation/Doctors):


In [ ]:
# ==========================================
# MASTER PIPELINE: Phase 3 (Reverse)
# (Standard Hindi Doctor -> Bhojpuri Patient)
# ==========================================
!pip install datasets soundfile transformers peft torch torchaudio librosa deep-translator gTTS
import os
import time
import torch
import torchaudio
import librosa
import warnings
from datasets import load_dataset
from deep_translator import GoogleTranslator
from gtts import gTTS
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from google.colab import drive, userdata

warnings.filterwarnings("ignore")
drive.mount('/content/drive')

# 1. Setup Reverse Folders
source_dir = "/content/drive/MyDrive/Reverse_Source_Hindi/"
target_dir = "/content/drive/MyDrive/Reverse_Target_Bhojpuri/"
os.makedirs(source_dir, exist_ok=True)
os.makedirs(target_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Reverse Pipeline on: {device.upper()}")

# 2. Load Engines
hubert = torch.hub.load("bshall/hubert:main", "hubert_soft").to(device).eval()
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
whisper_model.eval()

# THE FLIP: Translate Hindi to Bhojpuri
text_translator = GoogleTranslator(source='hi', target='bho')
max_samples = 500
processed_count = 0

# 3. Stream Data (We will filter for Hindi files this time)
hf_token = userdata.get('HF_TOKEN')
stream = load_dataset("ARTPARK-IISc/Vaani", name="Bihar_Saran", split="train", streaming=True, token=hf_token)

print("\n--- INITIATING REVERSE END-TO-END PROCESSING ---")

for sample in stream:
    if processed_count >= max_samples:
        break

    # FLIP: We are looking for Hindi input now!
    if sample.get('language') == 'Hindi':
        target_check_path = os.path.join(target_dir, f"pair_{processed_count}.pt")
        if os.path.exists(target_check_path):
            processed_count += 1
            continue

        print(f"\n[Processing Reverse Pair ID: {processed_count}]")
        try:
            # --- STEP A: Extract Source Math (Hindi) ---
            audio_data = sample['audio']['array']
            sr = sample['audio']['sampling_rate']
            source_wave = torch.tensor(audio_data).float().to(device)
            if sr != 16000:
                source_wave = torchaudio.functional.resample(source_wave, sr, 16000)

            with torch.no_grad():
                source_units = hubert.units(source_wave.unsqueeze(0).unsqueeze(0)).squeeze(0).cpu()
            torch.save(source_units.clone(), os.path.join(source_dir, f"pair_{processed_count}.pt"))

            # --- STEP B: ASR Bridge ---
            audio_np = source_wave.squeeze().cpu().numpy()
            inputs = whisper_processor(audio_np, sampling_rate=16000, return_tensors="pt")

            with torch.no_grad():
                predicted_ids = whisper_model.generate(
                    inputs.input_features.to(device), language="hindi", task="transcribe",
                    repetition_penalty=1.2, no_repeat_ngram_size=2, max_new_tokens=60
                )

            heard_text = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
            if len(heard_text) < 3: continue

            # --- STEP C: Translate to Bhojpuri ---
            bhojpuri_text = text_translator.translate(heard_text)
            print(f"  -> Translated to Bhojpuri: '{bhojpuri_text}'")

            # --- STEP D: Synthesize Target Audio (Bhojpuri text via Hindi TTS engine) ---
            temp_tts_path = f"temp_bhojpuri_{processed_count}.mp3"
            tts = gTTS(text=bhojpuri_text, lang='hi', slow=False)
            tts.save(temp_tts_path)

            # --- STEP E: Extract Target Math (Bhojpuri) ---
            y, sr_tts = librosa.load(temp_tts_path, sr=16000)
            target_wave = torch.tensor(y).unsqueeze(0).unsqueeze(0).float().to(device)

            with torch.no_grad():
                target_units = hubert.units(target_wave).squeeze(0).cpu()
            torch.save(target_units.clone(), target_check_path)

            os.remove(temp_tts_path)
            print(f"  [SUCCESS] Reverse Pair {processed_count} securely saved.")
            processed_count += 1

        except Exception as e:
            print(f"  [ERROR] Pipeline failed on Pair {processed_count}: {e}")

print(f"\nEnd-to-End Reverse Pipeline Complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Booting Reverse Pipeline on: CUDA
Downloading: "https://github.com/bshall/hubert/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/bshall/hubert/releases/download/v0.2/hubert-soft-35d9f29f.pt" to /root/.cache/torch/hub/checkpoints/hubert-soft-35d9f29f.pt


100%|██████████| 361M/361M [00:04<00:00, 92.1MB/s]


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

  [SUCCESS] Reverse Pair 496 securely saved.

[Processing Reverse Pair ID: 497]
  -> Translated to Bhojpuri: 'एकरा के देखत नाविक के स्थायी बना दिहल गईल बा, इ कहि के कईल जाता'


Both `max_new_tokens` (=60) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [SUCCESS] Reverse Pair 497 securely saved.

[Processing Reverse Pair ID: 498]
  -> Translated to Bhojpuri: 'एह पेड़ के फूल लउकेला आ लहर अन्हार बा।'


Both `max_new_tokens` (=60) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [SUCCESS] Reverse Pair 498 securely saved.

[Processing Reverse Pair ID: 499]
  -> Translated to Bhojpuri: 'मछरी के बाजार बा, ओहिजा रहे के कई गो तरीका बा।'
  [SUCCESS] Reverse Pair 499 securely saved.

End-to-End Reverse Pipeline Complete!


In [ ]:
# ==========================================
# PHASE 4: ACADEMIC BENCHMARKING (RTF & ASR-Text)
# ==========================================
import time
import torch
import torchaudio
import torch.nn as nn
from transformers import T5Model, WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from datasets import load_dataset
import warnings

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Benchmarking Suite on: {device.upper()}...\n")

# 1. Load the S2S Architecture (Bhojpuri -> Hindi Brain)
hubert = torch.hub.load("bshall/hubert:main", "hubert_soft").to(device).eval()
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft").to(device).eval()
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_soft").to(device).eval()

weights_dir = "/content/drive/MyDrive/S2S_LoRA_Weights/"
base_model = T5Model.from_pretrained("t5-small")
translator = PeftModel.from_pretrained(base_model, weights_dir).to(device).eval()

t5_dim = base_model.config.d_model
input_bridge = nn.Linear(256, t5_dim).to(device)
output_bridge = nn.Linear(t5_dim, 256).to(device)
input_bridge.load_state_dict(torch.load(weights_dir + "projection.pt", map_location=device, weights_only=True))
output_bridge.load_state_dict(torch.load(weights_dir + "output_bridge.pt", map_location=device, weights_only=True))
input_bridge.eval()
output_bridge.eval()

# 2. Load Evaluation ASR (To transcribe the output for semantic scoring)
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).eval()

# 3. Stream Unseen Test Data
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
test_stream = load_dataset("ARTPARK-IISc/Vaani", name="Bihar_Saran", split="train", streaming=True, token=hf_token)

test_samples = []
for i, sample in enumerate(test_stream):
    if sample.get('language') == 'Bhojpuri' and i > 505: # Skip training data
        test_samples.append(sample)
    if len(test_samples) == 5:
        break

print("--- COMMENCING LATENCY & ACCURACY BENCHMARK ---")
total_audio_duration = 0
total_processing_time = 0

for idx, sample in enumerate(test_samples):
    print(f"\n[Evaluating Sample {idx+1}/5]")
    audio_data = torch.tensor(sample['audio']['array']).float().to(device)
    sr = sample['audio']['sampling_rate']
    if sr != 16000:
        audio_data = torchaudio.functional.resample(audio_data, sr, 16000)

    # Calculate Audio Duration
    duration = len(audio_data) / 16000
    total_audio_duration += duration
    print(f"  🎙️ Input Duration: {duration:.2f} seconds")

    # --- START TIMING THE ARCHITECTURE ---
    start_time = time.time()

    with torch.no_grad():
        # THE FIX: Hugging Face array is 1D [Time]. HuBERT needs [Batch, Channel, Time].
        # We unsqueeze TWICE to make it [1, 1, Time]
        source_units = hubert.units(audio_data.unsqueeze(0).unsqueeze(0))

        proj_src = input_bridge(source_units)
        outputs = translator(inputs_embeds=proj_src, decoder_inputs_embeds=proj_src)
        target_math = output_bridge(outputs.last_hidden_state)

        mel = acoustic.generate(target_math)
        mel = mel.transpose(1, 2)
        synthesized_wav = hifigan(mel).squeeze()

    end_time = time.time()
    # --- END TIMING ---

    process_time = end_time - start_time
    total_processing_time += process_time
    print(f"  ⚡ Inference Time: {process_time:.2f} seconds")

    # ASR Evaluation: What did the AI actually say?
    audio_np = synthesized_wav.cpu().numpy()
    inputs = whisper_processor(audio_np, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        pred_ids = whisper_model.generate(
            inputs.input_features.to(device),
            language="hindi",
            task="transcribe",
            repetition_penalty=1.2,
            no_repeat_ngram_size=2
        )
    transcription = whisper_processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()

    print(f"  📝 Final Translated Output (ASR): '{transcription}'")

# --- FINAL METRICS ---
rtf = total_processing_time / total_audio_duration
print("\n=============================================")
print("🏆 FINAL BENCHMARK METRICS (S2S Architecture)")
print("=============================================")
print(f"Total Audio Processed : {total_audio_duration:.2f} seconds")
print(f"Total Compute Time    : {total_processing_time:.2f} seconds")
print(f"Real-Time Factor (RTF): {rtf:.3f}")
if rtf < 1.0:
    print("Verdict: FASTER THAN REAL-TIME. Clinically Viable. ✅")
else:
    print("Verdict: SLOWER THAN REAL-TIME. Requires Optimization. ⚠️")
print("=============================================")

Booting Benchmarking Suite on: CUDA...



Using cache found in /root/.cache/torch/hub/bshall_hubert_main
Using cache found in /root/.cache/torch/hub/bshall_acoustic-model_main
Using cache found in /root/.cache/torch/hub/bshall_hifigan_main


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

...

--- COMMENCING LATENCY & ACCURACY BENCHMARK ---

[Evaluating Sample 1/5]
  🎙️ Input Duration: 11.01 seconds


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  ⚡ Inference Time: 1.78 seconds


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


  📝 Final Translated Output (ASR): 'अईनम्र बेगे है, पलाक सर्ही लिंय दीची कुतर of इत भिड़द वो नमभएट. और की सनीं तनिन ये आप उछोः फेल जझी मूऊँइ खेत कोतब थौखान सेरा रहें'

[Evaluating Sample 2/5]
  🎙️ Input Duration: 9.84 seconds
  ⚡ Inference Time: 0.86 seconds
  📝 Final Translated Output (ASR): 'याग्छि मुरात हें ताम एक दान है, जहाल पल हरी आनिं ही कासे, ॐ'

[Evaluating Sample 3/5]
  🎙️ Input Duration: 6.52 seconds
  ⚡ Inference Time: 0.80 seconds
  📝 Final Translated Output (ASR): 'इंके मी लान हैंद at, वुत्स रो केम जो पूछ खिल आयें और बेहॉपोफृ चरे हेग.'

[Evaluating Sample 4/5]
  🎙️ Input Duration: 11.54 seconds
  ⚡ Inference Time: 1.34 seconds
  📝 Final Translated Output (ASR): 'यह कुई पानी कर्तिके बोंडाय, भर सक टीदर जर है. वमबसे कोगे, । फली ल़ा आया में दूलाइ नी हॉए. मक्यन थीरा, मृझारीयम ठाहा हमेशा.'

[Evaluating Sample 5/5]
  🎙️ Input Duration: 3.94 seconds
  ⚡ Inference Time: 0.38 seconds
  📝 Final Translated Output (ASR): 'अद्रानी कोलते हैं।'

🏆 FINAL BENCHMARK METRICS (S2S Architectur

In [ ]:
# ==========================================
# PHASE 4.5: THE COMPARATIVE BENCHMARK
# (S2S Architecture vs. Cascaded Baseline)
# ==========================================
!pip install gTTS
import time
import torch
import torchaudio
import torch.nn as nn
from transformers import T5Model, WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from datasets import load_dataset
from deep_translator import GoogleTranslator
from gtts import gTTS
import os
import warnings

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Comparative Benchmark on: {device.upper()}...\n")

# 1. Load S2S Models
hubert = torch.hub.load("bshall/hubert:main", "hubert_soft").to(device).eval()
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft").to(device).eval()
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_soft").to(device).eval()

weights_dir = "/content/drive/MyDrive/S2S_LoRA_Weights/"
base_model = T5Model.from_pretrained("t5-small")
translator = PeftModel.from_pretrained(base_model, weights_dir).to(device).eval()

t5_dim = base_model.config.d_model
input_bridge = nn.Linear(256, t5_dim).to(device)
output_bridge = nn.Linear(t5_dim, 256).to(device)
input_bridge.load_state_dict(torch.load(weights_dir + "projection.pt", map_location=device, weights_only=True))
output_bridge.load_state_dict(torch.load(weights_dir + "output_bridge.pt", map_location=device, weights_only=True))
input_bridge.eval()
output_bridge.eval()

# 2. Load Cascaded Models
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).eval()
text_translator = GoogleTranslator(source='bho', target='hi')

# 3. Stream 3 Unseen Test Samples
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
test_stream = load_dataset("ARTPARK-IISc/Vaani", name="Bihar_Saran", split="train", streaming=True, token=hf_token)

test_samples = []
for i, sample in enumerate(test_stream):
    if sample.get('language') == 'Bhojpuri' and i > 510:
        test_samples.append(sample)
    if len(test_samples) == 3: # Testing 3 for brevity
        break

print("--- COMMENCING HEAD-TO-HEAD BENCHMARK ---")

for idx, sample in enumerate(test_samples):
    print(f"\n[Evaluating Sample {idx+1}/3]")
    audio_data = torch.tensor(sample['audio']['array']).float().to(device)
    sr = sample['audio']['sampling_rate']
    if sr != 16000:
        audio_data = torchaudio.functional.resample(audio_data, sr, 16000)

    audio_np = audio_data.cpu().numpy()
    duration = len(audio_data) / 16000
    print(f"  🎙️ Input Duration: {duration:.2f} seconds")

    # ==========================================
    # PATH A: S2S ARCHITECTURE
    # ==========================================
    start_time_s2s = time.time()
    with torch.no_grad():
        source_units = hubert.units(audio_data.unsqueeze(0).unsqueeze(0))
        proj_src = input_bridge(source_units)
        outputs = translator(inputs_embeds=proj_src, decoder_inputs_embeds=proj_src)
        target_math = output_bridge(outputs.last_hidden_state)
        mel = acoustic.generate(target_math)
        mel = mel.transpose(1, 2)
        s2s_wav = hifigan(mel).squeeze()
    time_s2s = time.time() - start_time_s2s

    # Evaluate S2S ASR
    inputs = whisper_processor(s2s_wav.cpu().numpy(), sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        pred_ids = whisper_model.generate(inputs.input_features.to(device), language="hindi", task="transcribe")
    s2s_transcription = whisper_processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()

    # ==========================================
    # PATH B: CASCADED BASELINE (Google API)
    # ==========================================
    start_time_casc = time.time()
    inputs_casc = whisper_processor(audio_np, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        pred_ids_casc = whisper_model.generate(inputs_casc.input_features.to(device), language="hindi", task="transcribe")
    heard_bhojpuri = whisper_processor.batch_decode(pred_ids_casc, skip_special_tokens=True)[0].strip()
    clean_hindi = text_translator.translate(heard_bhojpuri)
    tts = gTTS(text=clean_hindi, lang='hi', tld='co.in', slow=False)
    tts.save("cascaded_temp.mp3")
    time_casc = time.time() - start_time_casc

    print(f"  [S2S Model] RTF: {(time_s2s/duration):.3f} | ASR Output: '{s2s_transcription}'")
    print(f"  [Cascaded ] RTF: {(time_casc/duration):.3f} | Text Output: '{clean_hindi}'")

print("\n=============================================")
print("✅ BENCHMARK COMPLETE. Ready for deployment.")
print("=============================================")

Booting Comparative Benchmark on: CUDA...



Using cache found in /root/.cache/torch/hub/bshall_hubert_main
Using cache found in /root/.cache/torch/hub/bshall_acoustic-model_main
Using cache found in /root/.cache/torch/hub/bshall_hifigan_main


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

--- COMMENCING HEAD-TO-HEAD BENCHMARK ---

[Evaluating Sample 1/3]
  🎙️ Input Duration: 11.54 seconds


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

  [S2S Model] RTF: 0.380 | ASR Output: 'यह पानी का खंकी बून्डाः पर असा टीदर जरह वा है बमपो से पुए टीदर बूनाड़ है भी दूलाई नी वाई बाकि अन थी अद भिटन लवर आयाम पे देर पोड़े दिखे रे आस्वान पिखे रे'
  [Cascaded ] RTF: 0.832 | Text Output: 'करने के लिए करने के लिए करने के लिए करने के लिए करने के लिए करने के लिए करने के लिए करने के लिए करने के लिए करने के लिए'

[Evaluating Sample 2/3]
  🎙️ Input Duration: 3.94 seconds
  [S2S Model] RTF: 0.105 | ASR Output: 'अद्वाँ प्रद्वाँ प्रद्वाँ प्रद्वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वाँ वा'
  [Cascaded ] RTF: 0.962 | Text Output: 'एड्री ब्रेड पर वोट है जो हरे और पीले रंग की है'

[Evaluating Sample 3/3]
  🎙️ Input Duration: 3.58 seconds
  [S2S Model] RTF: 0.101 | ASR Output: 'अग,

In [ ]:
# ==========================================
# DATA ACQUISITION: Download Unseen Test Audio
# (For the UI Demo)
# ==========================================
!pip install datasets soundfile

import soundfile as sf
from datasets import load_dataset
from google.colab import files, userdata
import warnings

warnings.filterwarnings("ignore")

print("Fetching an unseen clinical audio file from the Vaani Dataset...")

# Load the dataset in streaming mode
hf_token = userdata.get('HF_TOKEN')
test_stream = load_dataset(
    "ARTPARK-IISc/Vaani",
    name="Bihar_Saran",
    split="train",
    streaming=True,
    token=hf_token
)

# Skip the 500 training files + the handful used for benchmarking.
target_index = 525
current_bhojpuri_count = 0
output_filename = f"unseen_patient_{target_index}.wav"

for sample in test_stream:
    if sample.get('language') == 'Bhojpuri':
        if current_bhojpuri_count < target_index:
            current_bhojpuri_count += 1
            continue

        # Found our target unseen file!
        audio_array = sample['audio']['array']
        sr = sample['audio']['sampling_rate']

        # Save it locally in Colab
        sf.write(output_filename, audio_array, sr)

        print(f"✅ Audio secured! (Original Ground Truth Text: '{sample.get('text')}')")
        print(f"💾 Triggering browser download for '{output_filename}'...")
        break

# Trigger the browser to download the file to local
files.download(output_filename)
print("\n⬇️ Download started! You can now upload this file to your Gradio UI.")

Fetching an unseen clinical audio file from the Vaani Dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

✅ Audio secured! (Original Ground Truth Text: 'None')
💾 Triggering browser download for 'unseen_patient_525.wav'...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️ Download started! You can now upload this file to your Gradio UI.


In [ ]:
# ==========================================
# Evaluation UI
# (Live Head-to-Head: S2S vs Cascaded)
# ==========================================
!pip install gradio transformers peft torch torchaudio librosa deep-translator gTTS

import gradio as gr
import time
import torch
import torchaudio
import torch.nn as nn
from transformers import T5Model, WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from deep_translator import GoogleTranslator
from gtts import gTTS
import warnings

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Defense Day UI on: {device.upper()}...")

# 1. Load S2S Models
hubert = torch.hub.load("bshall/hubert:main", "hubert_soft").to(device).eval()
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft").to(device).eval()
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_soft").to(device).eval()

weights_dir = "/content/drive/MyDrive/S2S_LoRA_Weights/"
base_model = T5Model.from_pretrained("t5-small")
translator = PeftModel.from_pretrained(base_model, weights_dir).to(device).eval()

t5_dim = base_model.config.d_model
input_bridge = nn.Linear(256, t5_dim).to(device)
output_bridge = nn.Linear(t5_dim, 256).to(device)
input_bridge.load_state_dict(torch.load(weights_dir + "projection.pt", map_location=device, weights_only=True))
output_bridge.load_state_dict(torch.load(weights_dir + "output_bridge.pt", map_location=device, weights_only=True))
input_bridge.eval()
output_bridge.eval()

# 2. Load Cascaded Models
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).eval()
text_translator = GoogleTranslator(source='bho', target='hi')

# 3. The Live Evaluation Function (WITH SAFETY FIXES)
def live_benchmark(audio_path):
    if audio_path is None:
        return None, None, "No audio provided.", "No audio provided.", "0.00s", "0.00s"

    wav, sr = torchaudio.load(audio_path)

    # SAFETY FIX 1: Force Mono Audio (In case the upload is Stereo)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    wav = wav.float().to(device)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    audio_np = wav.squeeze().cpu().numpy()
    duration = len(wav.squeeze()) / 16000

    # --- PATH A: S2S Architecture ---
    start_s2s = time.time()
    with torch.no_grad():
        # SAFETY FIX 2: torchaudio gives [1, Time]. We only need ONE unsqueeze for Batch.
        source_units = hubert.units(wav.unsqueeze(0))
        proj_src = input_bridge(source_units)
        outputs = translator(inputs_embeds=proj_src, decoder_inputs_embeds=proj_src)
        target_math = output_bridge(outputs.last_hidden_state)
        mel = acoustic.generate(target_math).transpose(1, 2)
        s2s_wav = hifigan(mel).squeeze()

    s2s_time = time.time() - start_s2s
    s2s_audio_path = "s2s_output.wav"
    torchaudio.save(s2s_audio_path, s2s_wav.unsqueeze(0).cpu(), 16000)

    # Evaluate S2S ASR
    inputs = whisper_processor(s2s_wav.cpu().numpy(), sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        pred_ids = whisper_model.generate(
            inputs.input_features.to(device),
            language="hindi",
            task="transcribe",
            repetition_penalty=1.2,
            no_repeat_ngram_size=2
        )
    s2s_text = whisper_processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()

    # --- PATH B: Cascaded Baseline ---
    start_casc = time.time()
    inputs_casc = whisper_processor(audio_np, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        pred_ids_casc = whisper_model.generate(
            inputs_casc.input_features.to(device),
            language="hindi",
            task="transcribe"
        )
    heard_text = whisper_processor.batch_decode(pred_ids_casc, skip_special_tokens=True)[0].strip()
    casc_text = text_translator.translate(heard_text)

    casc_audio_path = "casc_output.mp3"
    tts = gTTS(text=casc_text, lang='hi', tld='co.in', slow=False)
    tts.save(casc_audio_path)
    casc_time = time.time() - start_casc

    # Calculate RTF
    rtf_s2s = f"{(s2s_time/duration):.3f} ({(duration/s2s_time):.1f}x Faster)" if s2s_time > 0 else "0.000"
    rtf_casc = f"{(casc_time/duration):.3f}"

    return s2s_audio_path, casc_audio_path, s2s_text, casc_text, rtf_s2s, rtf_casc

# 4. The Defense UI Layout
with gr.Blocks(theme=gr.themes.Monochrome()) as defense_app:
    gr.Markdown("# 🔬 M.Tech Defense: Architecture Evaluation Panel")
    gr.Markdown("Test the Experimental **Discrete S2S Model** against the traditional **Cascaded Baseline** in real-time.")

    with gr.Row():
        input_audio = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Input: Rural Bhojpuri Patient")

    eval_btn = gr.Button("Run Live Benchmark", variant="primary")

    with gr.Row():
        # Column 1: Your Architecture
        with gr.Column(variant="panel"):
            gr.Markdown("### ⚙️ Proposed S2S Architecture (Textless)")
            s2s_out = gr.Audio(label="Synthesized Math (Hindi Output)")
            s2s_rtf = gr.Textbox(label="Speed (Real-Time Factor)")
            s2s_txt = gr.Textbox(label="Whisper ASR Evaluation (Notice the Vocoder penalty)")

        # Column 2: The Baseline
        with gr.Column(variant="panel"):
            gr.Markdown("###  Cascaded Baseline (Google API)")
            casc_out = gr.Audio(label="TTS Output")
            casc_rtf = gr.Textbox(label="Speed (Real-Time Factor)")
            casc_txt = gr.Textbox(label="Text Translation (Watch for Hallucinations)")

    eval_btn.click(
        fn=live_benchmark,
        inputs=input_audio,
        outputs=[s2s_out, casc_out, s2s_txt, casc_txt, s2s_rtf, casc_rtf]
    )

defense_app.launch(share=True)

INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
  Attempting uninstall: typer
    Found existing installation: typer 0.24.1
    Uninstalling typer-0.24.1:
      Successfully uninstalled typer-0.24.1
  Attempting uninstall: typer-slim
    Found existing installation: typer-slim 0.24.0
    Uninstalling typer-slim-0.24.0:
      Successfully uninstalled typer-slim-0.24.0
Bootin

100%|██████████| 361M/361M [00:09<00:00, 38.0MB/s]


Downloading: "https://github.com/bshall/acoustic-model/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/bshall/acoustic-model/releases/download/v0.1/hubert-soft-0321fd7e.pt" to /root/.cache/torch/hub/checkpoints/hubert-soft-0321fd7e.pt


100%|██████████| 71.8M/71.8M [00:01<00:00, 43.3MB/s]


Downloading: "https://github.com/bshall/hifigan/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/bshall/hifigan/releases/download/v0.1/hifigan-hubert-soft-65f03469.pt" to /root/.cache/torch/hub/checkpoints/hifigan-hubert-soft-65f03469.pt


...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4fe1143323880c0f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==========================================
# PHASE 6: DATASET B TRAINING LOOP
# (Training the Hindi -> Bhojpuri Brain)
# ==========================================
!pip install transformers peft torch

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import T5Model
from peft import get_peft_model, LoraConfig, TaskType
from google.colab import drive

drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Reverse Training on: {device.upper()}")

# 1. Define the Reverse Dataset
class ReverseClinicalDataset(Dataset):
    def __init__(self, source_dir, target_dir):
        self.source_dir = source_dir
        self.target_dir = target_dir
        self.files = [f for f in os.listdir(source_dir) if f.endswith('.pt')]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]
        source_units = torch.load(os.path.join(self.source_dir, file_name), weights_only=True)
        target_units = torch.load(os.path.join(self.target_dir, file_name), weights_only=True)

        # Ensure dimensions match [Time, 256]
        if source_units.dim() == 3: source_units = source_units.squeeze()
        if target_units.dim() == 3: target_units = target_units.squeeze()

        return source_units.to(device), target_units.to(device)

source_path = "/content/drive/MyDrive/Reverse_Source_Hindi/"
target_path = "/content/drive/MyDrive/Reverse_Target_Bhojpuri/"
dataset = ReverseClinicalDataset(source_path, target_path)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# 2. Initialize T5 and LoRA
print("Initializing Base T5 Model...")
base_model = T5Model.from_pretrained("t5-small").to(device)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)
translator = get_peft_model(base_model, lora_config).to(device)

# 3. Initialize Dimensional Bridges
t5_dim = base_model.config.d_model
input_bridge = nn.Linear(256, t5_dim).to(device)
output_bridge = nn.Linear(t5_dim, 256).to(device)

# 4. Training Setup
optimizer = torch.optim.AdamW(
    list(translator.parameters()) + list(input_bridge.parameters()) + list(output_bridge.parameters()),
    lr=1e-4
)
loss_fn = nn.MSELoss()

# 5. The Training Loop (5 Epochs)
epochs = 5
print("\nStarting Reverse LoRA Training (Hindi -> Bhojpuri)...")

for epoch in range(epochs):
    translator.train()
    total_loss = 0

    for batch_idx, (source, target) in enumerate(dataloader):
        optimizer.zero_grad()

        # THE FIX: DataLoader already provides [Batch, Time, 256].
        # We pass it straight to the bridge!
        proj_src = input_bridge(source)
        outputs = translator(inputs_embeds=proj_src, decoder_inputs_embeds=proj_src)
        pred_target = output_bridge(outputs.last_hidden_state)

        # Dynamic Time Warping / Padding fix for mismatched sequence lengths
        min_len = min(pred_target.size(1), target.size(1))
        loss = loss_fn(pred_target[:, :min_len, :], target[:, :min_len, :])

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(dataloader)} | MSE Loss: {loss.item():.4f}")

    print(f"--- Epoch {epoch+1} Average Loss: {(total_loss/len(dataloader)):.4f} ---\n")

# 6. Save the Reverse Brain
save_dir = "/content/drive/MyDrive/Reverse_LoRA_Weights/"
os.makedirs(save_dir, exist_ok=True)

translator.save_pretrained(save_dir)
torch.save(input_bridge.state_dict(), save_dir + "projection.pt")
torch.save(output_bridge.state_dict(), save_dir + "output_bridge.pt")

print(f"SUCCESS! Reverse AI Brain saved to {save_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Booting Reverse Training on: CUDA
Initializing Base T5 Model...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


Starting Reverse LoRA Training (Hindi -> Bhojpuri)...
Epoch 1/5 | Batch 0/500 | MSE Loss: 0.3672
Epoch 1/5 | Batch 50/500 | MSE Loss: 0.1848
Epoch 1/5 | Batch 100/500 | MSE Loss: 0.1808
Epoch 1/5 | Batch 150/500 | MSE Loss: 0.1815
Epoch 1/5 | Batch 200/500 | MSE Loss: 0.1662
Epoch 1/5 | Batch 250/500 | MSE Loss: 0.1681
Epoch 1/5 | Batch 300/500 | MSE Loss: 0.1707
Epoch 1/5 | Batch 350/500 | MSE Loss: 0.1656
Epoch 1/5 | Batch 400/500 | MSE Loss: 0.1657
Epoch 1/5 | Batch 450/500 | MSE Loss: 0.1697
--- Epoch 1 Average Loss: 0.1778 ---

Epoch 2/5 | Batch 0/500 | MSE Loss: 0.1697
Epoch 2/5 | Batch 50/500 | MSE Loss: 0.1700
Epoch 2/5 | Batch 100/500 | MSE Loss: 0.1649
Epoch 2/5 | Batch 150/500 | MSE Loss: 0.1661
Epoch 2/5 | Batch 200/500 | MSE Loss: 0.1677
Epoch 2/5 | Batch 250/500 | MSE Loss: 0.1695
Epoch 2/5 | Batch 300/500 | MSE Loss: 0.1702
Epoch 2/5 | Batch 350/500 | MSE Loss: 0.1686
Epoch 2/5 | Batch 400/500 | MSE Loss: 0.1661
Epoch 2/5 | Batch 450/500 | MSE Loss: 0.1745
--- Epoch 2 A

In [ ]:
# ==========================================
# PHASE 7: THE MASTER CLINICAL APPLICATION
# (Fully Bidirectional S2S Translation)
# ==========================================
!pip install gradio transformers peft torch torchaudio librosa nest-asyncio

# --- THE COLAB PATCH ---
import nest_asyncio
nest_asyncio.apply()
# -----------------------

import gradio as gr
import torch
import torchaudio
import torch.nn as nn
from transformers import T5Model
from peft import PeftModel
import warnings

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Master Clinical AI on: {device.upper()}...\n")

# 1. Load Universal Audio Models (Ear & Voice)
print("Loading Acoustic Infrastructure...")
hubert = torch.hub.load("bshall/hubert:main", "hubert_soft").to(device).eval()
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft").to(device).eval()
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_soft").to(device).eval()

# 2. Load Brain A (Patient -> Doctor)
print("Loading Forward Brain (Bhojpuri -> Hindi)...")
fwd_dir = "/content/drive/MyDrive/S2S_LoRA_Weights/"
base_fwd = T5Model.from_pretrained("t5-small").to(device)
brain_fwd = PeftModel.from_pretrained(base_fwd, fwd_dir).to(device).eval()

in_b_fwd = nn.Linear(256, 512).to(device)
out_b_fwd = nn.Linear(512, 256).to(device)
in_b_fwd.load_state_dict(torch.load(fwd_dir + "projection.pt", map_location=device, weights_only=True))
out_b_fwd.load_state_dict(torch.load(fwd_dir + "output_bridge.pt", map_location=device, weights_only=True))
in_b_fwd.eval()
out_b_fwd.eval()

# 3. Load Brain B (Doctor -> Patient)
print("Loading Reverse Brain (Hindi -> Bhojpuri)...")
rev_dir = "/content/drive/MyDrive/Reverse_LoRA_Weights/"
base_rev = T5Model.from_pretrained("t5-small").to(device)
brain_rev = PeftModel.from_pretrained(base_rev, rev_dir).to(device).eval()

in_b_rev = nn.Linear(256, 512).to(device)
out_b_rev = nn.Linear(512, 256).to(device)
in_b_rev.load_state_dict(torch.load(rev_dir + "projection.pt", map_location=device, weights_only=True))
out_b_rev.load_state_dict(torch.load(rev_dir + "output_bridge.pt", map_location=device, weights_only=True))
in_b_rev.eval()
out_b_rev.eval()

# 4. Core Routing Logic
def bidirectional_translate(audio_path, direction):
    if audio_path is None:
        return None

    wav, sr = torchaudio.load(audio_path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True) # Force Mono

    wav = wav.float().to(device)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    # Route based on UI Toggle
    if direction == "Patient ➔ Doctor (Bhojpuri to Hindi)":
        active_brain = brain_fwd
        active_in = in_b_fwd
        active_out = out_b_fwd
    else:
        active_brain = brain_rev
        active_in = in_b_rev
        active_out = out_b_rev

    with torch.no_grad():
        source_units = hubert.units(wav.unsqueeze(0))
        proj_src = active_in(source_units)
        outputs = active_brain(inputs_embeds=proj_src, decoder_inputs_embeds=proj_src)
        target_math = active_out(outputs.last_hidden_state)
        mel = acoustic.generate(target_math).transpose(1, 2)
        synthesized_wav = hifigan(mel).squeeze()

    output_path = "final_clinical_output.wav"
    torchaudio.save(output_path, synthesized_wav.unsqueeze(0).cpu(), 16000)
    return output_path

# 5. Master UI Layout
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as master_app:
    gr.Markdown("# 🏥 Bidirectional Clinical Translator (S2S)")
    gr.Markdown("### Zero-Shot, Textless Medical Translation Architecture")
    gr.Markdown("*Developed by Khushi & Surbhi | IIT Jodhpur*")

    with gr.Row():
        with gr.Column():
            direction_toggle = gr.Radio(
                choices=["Patient ➔ Doctor (Bhojpuri to Hindi)", "Doctor ➔ Patient (Hindi to Bhojpuri)"],
                value="Patient ➔ Doctor (Bhojpuri to Hindi)",
                label="Select Translation Direction",
            )
            gr.Markdown("### 🎤 Audio Input")
            audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Record or Upload Voice")
            translate_btn = gr.Button("Translate Audio", variant="primary")

        with gr.Column():
            gr.Markdown("### 🎧 AI Synthesized Output")
            audio_output = gr.Audio(label="Translated Speech", interactive=False)

    translate_btn.click(
        fn=bidirectional_translate,
        inputs=[audio_input, direction_toggle],
        outputs=audio_output
    )

print("\nSystem Online. Launching Interface...")
# --- THE DEBUG PATCH ---
master_app.launch(share=True, debug=False)
# -----------------------

Booting Master Clinical AI on: CUDA...

Loading Acoustic Infrastructure...


Using cache found in /root/.cache/torch/hub/bshall_hubert_main
Using cache found in /root/.cache/torch/hub/bshall_acoustic-model_main
Using cache found in /root/.cache/torch/hub/bshall_hifigan_main


Loading Forward Brain (Bhojpuri -> Hindi)...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading Reverse Brain (Hindi -> Bhojpuri)...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


System Online. Launching Interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://01855256a8a105437a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
